# Training

This notebook puts everything together: data, model, loss, optimizer, and the
training loop. We'll train a classifier end-to-end on a tiny dataset.

In PyTorch you write a manual `for epoch in range(N)` loop with `loss.backward()`
and `optimizer.step()`. In idris-ml, the `fit` driver handles all of that — and the
model is a **linear resource** threaded through `Control.Linear.LIO.L IO`, so you
literally cannot reuse a stale handle after training (it's a compile error).

## Optimizers

Four IO constructors over `OptimOpts` (`defaultOpts` = PyTorch defaults; record-update
to override betas/eps/clip). The backward pass, gradient clipping, and parameter update
happen in one fused step.

In [ ]:
:t sgd

In [ ]:
:t adamW

In [ ]:
:t rmsprop

## The `fit` driver

`fitSupervised` is the one driver for supervised training. It takes:
1. an **optimizer**,
2. a **loss function** that consumes the linear model, forwards it, and returns the
   scalar loss (banged) beside the rebuilt model,
3. a **`DataStream`** of batched `(input, target)` tensors,
4. a **`TrainConfig`** (epochs, early stopping),
5. the **initial model** (consumed linearly).

It returns `(trainedModel, epochsDone, finalLoss)`.

In [ ]:
:t fitSupervised

In [ ]:
:t simpleConfig

In [ ]:
:t patienceConfig

## One real gradient step

The full `fit` loop needs a `DataStream` + loss-fn plumbing that's awkward in a single
REPL cell (see the compiled example below for the whole thing). But we *can* run one
real SGD step live — build a one-layer model, forward a 3×2 batch, compute the NLL loss,
and `trainStep` it. The model is built with `runInitL` (which registers its params) and
threaded linearly through `forward`:

In [ ]:
:exec run (do {
  model <- runInitL (linear {i=2} {o=3} {ex=TapeExecutor} {dt=F64} {g=WithGrad});
  opt <- liftIO1 (sgd 0.1 defaultOpts);
  x <- liftIO1 (tensor {dims=[3,2]} {ex=TapeExecutor} {dt=F64} (FromVect [0.0,0.0, 1.0,1.0, 2.0,0.0]));
  y <- liftIO1 (tensor {dims=[3,3]} {ex=TapeExecutor} {dt=F64} (FromVect [1.0,0.0,0.0, 0.0,1.0,0.0, 0.0,0.0,1.0]));
  (MkBang out # model1) <- forward {b=3} model (retypeGrad x);
  loss <- tnllLossMeanL {b=3} {n=3} out (retypeGrad y);
  l <- liftIO1 (trainStep opt loss);
  discard model1;
  liftIO1 (putStrLn ("one SGD step done; loss = " ++ show l)) })

## Checkpointing and resume

`fit` can save every N epochs and resume automatically. Attach a `CheckpointPolicy` to the `TrainConfig` with `withCheckpoint`; the policy writes `<dir>/last.model.safetensors`, `.opt.safetensors`, and a `trainer_state.json` sidecar recording the epoch. When the directory already holds a `last` checkpoint, the next `fit` run resumes from it instead of starting over. `saveAll` / `load` are the manual surface underneath.

In [ ]:
:t withCheckpoint

In [ ]:
:t fileCheckpoint

In [ ]:
:t saveAll

In [ ]:
:t load

Train 3 epochs with a checkpoint every epoch (the `rm -rf` makes the cell re-runnable):

In [ ]:
:exec run (do {
  _ <- liftIO1 (system "rm -rf /tmp/idrisml-nb-checkpoint");
  model <- runInitL (linear {i=2} {o=3} {ex=TapeExecutor} {dt=F64} {g=WithGrad});
  opt <- liftIO1 (sgd 0.1 defaultOpts);
  (MkBang (epochs, finalLoss) # trained) <- fitSupervised opt (\m, xy => do { (MkBang out # m2) <- forward {b=3} m (retypeGrad (fst xy)); loss <- tnllLossMeanL {b=3} {n=3} out (retypeGrad (snd xy)); pure1 (MkBang loss # m2) }) (generate (do { x <- tensor {dims=[3,2]} {ex=TapeExecutor} {dt=F64} (FromVect [0.0,0.0, 1.0,1.0, 2.0,0.0]); y <- tensor {dims=[3,3]} {ex=TapeExecutor} {dt=F64} (FromVect [1.0,0.0,0.0, 0.0,1.0,0.0, 0.0,0.0,1.0]); pure (x, y) })) (withCheckpoint (fileCheckpoint "/tmp/idrisml-nb-checkpoint" 1 True opt) (simpleConfig 3)) model;
  discard trained;
  liftIO1 (putStrLn ("run 1: ran " ++ show epochs ++ " epochs; final loss " ++ show finalLoss)) })

Now ask for 5 epochs in the same directory. `fit` finds `last.trainer_state.json` at epoch 3 and resumes there, so only epochs 4 and 5 execute (the returned count is the cumulative epoch index):

In [ ]:
:exec run (do {
  model <- runInitL (linear {i=2} {o=3} {ex=TapeExecutor} {dt=F64} {g=WithGrad});
  opt <- liftIO1 (sgd 0.1 defaultOpts);
  (MkBang (epochs, finalLoss) # trained) <- fitSupervised opt (\m, xy => do { (MkBang out # m2) <- forward {b=3} m (retypeGrad (fst xy)); loss <- tnllLossMeanL {b=3} {n=3} out (retypeGrad (snd xy)); pure1 (MkBang loss # m2) }) (generate (do { x <- tensor {dims=[3,2]} {ex=TapeExecutor} {dt=F64} (FromVect [0.0,0.0, 1.0,1.0, 2.0,0.0]); y <- tensor {dims=[3,3]} {ex=TapeExecutor} {dt=F64} (FromVect [1.0,0.0,0.0, 0.0,1.0,0.0, 0.0,0.0,1.0]); pure (x, y) })) (withCheckpoint (fileCheckpoint "/tmp/idrisml-nb-checkpoint" 1 True opt) (simpleConfig 5)) model;
  discard trained;
  liftIO1 (putStrLn ("run 2: resumed and finished at epoch " ++ show epochs ++ "; final loss " ++ show finalLoss)) })

For real runs, the compiled examples wire the same policy to flags: `make example-rnn RNN_ARGS="--checkpoint-dir out --resume"`. The kernel buffers `:exec` output, so long training belongs there.

## End-to-end and compiled examples

The full `fitSupervised` run (data stream + loss fn + eval) is in
`packages/idris-ml-examples/src/Example/Supervised.idr`, and the text walkthrough is
[docs/getting-started.md](../../../../docs/getting-started.md). For real training
(thousands of epochs, larger models), use the compiled examples — the notebook kernel
buffers all output, making long runs impractical:

```bash
make example-supervised    # this notebook's task (1000 epochs)
make example-rnn           # RNN pattern prediction
make example-mnist         # CNN digit classification
make example-gpt           # character-level language model
```

All accept `--epochs`, `--lr`, and `--seed` flags.

## Training modes

Most work is supervised (`fitSupervised`). Recurrent / two-phase / RL training compose a
custom `EpochStep` and pass it to `fit`, or use the engine pieces directly:

| Scenario | Driver |
|----------|--------|
| Classification / regression | `fitSupervised` (pass a loss fn) |
| Mixed precision | `fitSupervisedMixed` (+ a `GradScaler`) |
| Recurrent / two-phase / RL | your own `EpochStep` → `fit` |

The next notebook covers the recurrent case.

## Early stopping

`simpleConfig n` runs exactly `n` epochs; `patienceConfig totalEpochs patience` stops when the loss stops improving.

Next: [05 Model Ownership](05_model_ownership.ipynb) — why a consumed model handle cannot be reused.